In [1]:
import os
import random

In [2]:
# Define the folder containing the audio files
audio_folder = r"AudioFiles/085"

# Get a list of all WAVE files in the folder
all_songs = [f for f in os.listdir(audio_folder) if f.endswith(".wav")]

In [4]:
k = 5  # songs to rank
random.shuffle(all_songs)

max_tests = len(all_songs) // (k + 1)
num_tests = min(10, max_tests)

test_sets = []
idx = 0

for _ in range(num_tests):
    reference = all_songs[idx]
    candidates = all_songs[idx + 1: idx + 1 + k]
    test_sets.append((reference, candidates))
    idx += k + 1

In [5]:
test_sets

[('085420.wav',
  ['085255.wav', '085843.wav', '085076.wav', '085073.wav', '085041.wav']),
 ('085038.wav',
  ['085869.wav', '085291.wav', '085342.wav', '085039.wav', '085599.wav']),
 ('085042.wav',
  ['085254.wav', '085072.wav', '085040.wav', '085027.wav', '085074.wav'])]

In [6]:
# Background image path
background_image = "background.png"

In [7]:
SUPABASE_URL = "https://zpvhnhnmjitkskhrhalp.supabase.co"
SUPABASE_ANON_KEY = "sb_publishable_jrzABfRbbZN4BNrUmABKag_b3LlgY18"

In [8]:
html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Music Similarity Test</title>

    <style>
        body {{
            font-family: Helvetica, Arial, sans-serif;
            background: url('{background_image}') no-repeat center center fixed;
            background-size: cover;
            color: white;
            max-width: 900px;
            margin: 0 auto;
            padding: 20px;
        }}

        input[type="text"] {{
            padding: 6px;
            width: 220px;
            margin-top: 4px;
            margin-bottom: 16px;
        }}

        audio {{
            display: block;
            margin-bottom: 12px;
        }}

        select {{
            margin-bottom: 12px;
            padding: 4px;
        }}

        .test-block {{
            margin-bottom: 30px;
            padding: 16px;
            background: rgba(0, 0, 0, 0.35);
            border-radius: 10px;
        }}

        .candidate-block {{
            margin-bottom: 18px;
            padding: 10px;
            background: rgba(255, 255, 255, 0.08);
            border-radius: 8px;
        }}

        .submit-row {{
            margin-top: 24px;
        }}
    </style>

    <script src="https://cdn.jsdelivr.net/npm/@supabase/supabase-js@2"></script>

    <script>
        const SUPABASE_URL = "{SUPABASE_URL}";
        const SUPABASE_ANON_KEY = "{SUPABASE_ANON_KEY}";
        const supabaseClient = supabase.createClient(SUPABASE_URL, SUPABASE_ANON_KEY);

        async function saveResults(event) {{
            event.preventDefault();

            const consentGiven = document.getElementById("consent").checked;
            if (!consentGiven) {{
                alert("You must agree to participate before submitting.");
                return;
            }}

            const participantCode = document.getElementById("participant_code")?.value.trim() || null;

            let results = [];

            for (let i = 0; i < {num_tests}; i++) {{
                const reference = document.getElementById("reference_" + i).dataset.song;
                let ranking = [];
                let usedRanks = [];

                for (let j = 0; j < {k}; j++) {{
                    const songEl = document.getElementById("song_" + i + "_" + j);
                    const rankEl = document.getElementById("rank_" + i + "_" + j);

                    const song = songEl.dataset.song;
                    const rank = parseInt(rankEl.value);

                    if (isNaN(rank)) {{
                        alert("Please assign all ranks in Test " + (i + 1) + ".");
                        return;
                    }}

                    usedRanks.push(rank);
                    ranking.push({{
                        song: song,
                        rank: rank
                    }});
                }}

                const uniqueRanks = new Set(usedRanks);
                if (uniqueRanks.size !== {k}) {{
                    alert("Each rank in Test " + (i + 1) + " must be unique.");
                    return;
                }}

                results.push({{
                    test: i + 1,
                    reference_song: reference,
                    ranking: ranking
                }});
            }}

            const submitBtn = document.getElementById("submitBtn");
            submitBtn.disabled = true;
            submitBtn.value = "Submitting...";

            const {{ error }} = await supabaseClient
                .from("listening_results")
                .insert([{{
                    participant_code: participantCode,
                    consent: true,
                    test_version: "v2_ranking",
                    responses: results
                }}]);

            submitBtn.disabled = false;
            submitBtn.value = "Save Results";

            if (error) {{
                console.error("Supabase insert error:", error);
                alert("Submission failed. Check console.");
                return;
            }}

            alert("Thank you. Your responses were saved.");
            document.querySelector("form").reset();
        }}
    </script>
</head>
<body>
    <h1>Music Similarity Test</h1>

    <p>
        Please rank the 5 songs from <b>most similar (1)</b> to <b>least similar ({k})</b> relative to the reference song.
    </p>

    <p>
        Participant code:<br>
        <input type="text" id="participant_code" placeholder="e.g. P001">
    </p>

    <form onsubmit="saveResults(event);">
"""

In [9]:
for i, (reference_song, candidates) in enumerate(test_sets):
    html_content += f"""
        <div class="test-block">
            <h3>Test {i + 1}</h3>

            <p><b>Reference Song:</b></p>
            <audio controls id="reference_{i}" data-song="{reference_song}">
                <source src="AudioFiles/085/{reference_song}" type="audio/wav">
            </audio>

            <p>Rank the songs below from most similar (1) to least similar ({k}):</p>
    """

    for j, song in enumerate(candidates):
        html_content += f"""
            <div class="candidate-block">
                <p><b>Song {j + 1}:</b></p>
                <audio controls id="song_{i}_{j}" data-song="{song}">
                    <source src="AudioFiles/085/{song}" type="audio/wav">
                </audio>

                <label for="rank_{i}_{j}">Rank:</label>
                <select id="rank_{i}_{j}">
                    <option value="">Select rank</option>
                    {''.join(f'<option value="{r}">{r}</option>' for r in range(1, k + 1))}
                </select>
            </div>
        """

    html_content += "</div>"

In [10]:
html_content += f"""
        <p>
            <input type="checkbox" id="consent" required>
            I agree to participate in this study and allow my responses to be used for research purposes.
        </p>

        <div class="submit-row">
            <input type="submit" id="submitBtn" value="Save Results">
        </div>
    </form>

    <p style="font-size: 12px; margin-top: 20px;">
        This experiment is based on research from
        <a href="https://arxiv.org/abs/1612.01840" target="_blank" style="color: white;">
            arXiv:1612.01840
        </a>.
    </p>

    <p style="font-size: 12px;">
        The metadata is released under the Creative Commons Attribution 4.0 International License (CC BY 4.0).
        We do not hold the copyright on the audio and distribute it under the license chosen by the artist.
        The dataset is meant for research purposes.
    </p>
</body>
</html>
"""

In [13]:
with open("index_dropdown.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("index_dropdown.html generated successfully.")

index_dropdown.html generated successfully.
